# Course 7: Privacy-Preserving Machine Learning
## Session 8: Computing on encrypted data

**Goal:** run encrypted inference with a toy additively homomorphic scheme written from scratch, measure the slowdown, and see why encrypted networks avoid ReLU.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. A toy Paillier cryptosystem
Paillier lets anyone add encrypted numbers and multiply an encrypted number by a plain one, without the secret key. That is enough for a linear layer with plain weights and encrypted inputs.

**This is a teaching toy.** The keys are short and the code is not constant-time. Never use it to protect real data. CKKS, used in the TenSEAL cell below, is a different scheme that also multiplies ciphertexts and packs vectors.

In [ ]:
import math, random, time
import numpy as np

def is_prime(n, rounds=30, rng=random.Random(1)):
    if n < 4: return n in (2, 3)
    if n % 2 == 0: return False
    d, s = n - 1, 0
    while d % 2 == 0: d //= 2; s += 1
    for _ in range(rounds):                               # Miller-Rabin test
        x = pow(rng.randrange(2, n - 1), d, n)
        if x in (1, n - 1): continue
        for _ in range(s - 1):
            x = x * x % n
            if x == n - 1: break
        else: return False
    return True

def random_prime(bits, rng):
    while True:
        p = rng.getrandbits(bits) | (1 << (bits - 1)) | 1
        if is_prime(p): return p

class Paillier:
    def __init__(self, bits=256, seed=0):
        rng = random.Random(seed)
        p, q = random_prime(bits, rng), random_prime(bits, rng)
        self.n, self.n2 = p * q, (p * q) ** 2
        self._lam = math.lcm(p - 1, q - 1); self._mu = pow(self._lam, -1, self.n)    # secret key
        self._rng = rng
    def encrypt(self, m):
        r = self._rng.randrange(1, self.n)
        return (1 + (m % self.n) * self.n) * pow(r, self.n, self.n2) % self.n2
    def decrypt(self, c):
        m = (pow(c, self._lam, self.n2) - 1) // self.n * self._mu % self.n
        return m - self.n if m > self.n // 2 else m          # map back to signed numbers
    def add(self, c1, c2): return c1 * c2 % self.n2          # Enc(a) * Enc(b) = Enc(a + b)
    def mul_plain(self, c, k): return pow(c, k % self.n, self.n2)   # Enc(a) ^ k = Enc(k a)

he = Paillier()
a, b = he.encrypt(17), he.encrypt(-5)
print("Enc(17) + Enc(-5) decrypts to", he.decrypt(he.add(a, b)), "| 3 * Enc(17) decrypts to", he.decrypt(he.mul_plain(a, 3)))
print("two encryptions of 17 look different:", he.encrypt(17) != he.encrypt(17))
assert he.decrypt(he.add(a, b)) == 12 and he.decrypt(he.mul_plain(a, 3)) == 51

### 2. Encrypted inference for a medical classifier
The patient encrypts their features. The hospital's server computes the model's score on the ciphertexts and returns an encrypted score. Only the patient can decrypt it. Real numbers are turned into whole numbers by fixed-point scaling.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
sc = StandardScaler().fit(Xtr); clf = LogisticRegression(max_iter=1000).fit(sc.transform(Xtr), ytr)
w, b0, Xs = clf.coef_[0], clf.intercept_[0], sc.transform(Xte)

S = 2 ** 10                                                    # fixed-point scale
w_int = [int(round(v * S)) for v in w]; b_int = int(round(b0 * S * S))

def encrypted_score(x):
    enc_x = [he.encrypt(int(round(v * S))) for v in x]         # patient side
    acc = he.encrypt(b_int)                                    # server side: only add and mul_plain
    for c, k in zip(enc_x, w_int): acc = he.add(acc, he.mul_plain(c, k))
    return he.decrypt(acc) / (S * S)                           # patient side

t0 = time.perf_counter(); enc_scores = np.array([encrypted_score(x) for x in Xs[:20]]); t_enc = (time.perf_counter() - t0) / 20
t0 = time.perf_counter()
for _ in range(2000): plain_scores = Xs[:20] @ w + b0
t_plain = (time.perf_counter() - t0) / 2000 / 20
print("largest difference between encrypted and plain scores: %.5f" % np.abs(enc_scores - plain_scores).max())
print("same predictions:", np.array_equal(enc_scores > 0, plain_scores > 0))
print(f"time per patient: encrypted {t_enc * 1000:.1f} ms | plain {t_plain * 1e6:.2f} microseconds | slowdown about {t_enc / t_plain:,.0f}x")
assert np.abs(enc_scores - plain_scores).max() < 0.05

### 3. Why encrypted networks avoid ReLU
Schemes like CKKS can add and multiply, so they can evaluate polynomials. ReLU is not a polynomial. A common fix is to train with a square activation instead. How much accuracy does that cost on a small task?

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits
d = load_digits(); Xd = torch.tensor(d.data / 16.0, dtype=torch.float32); yd = torch.tensor(d.target)
perm = torch.randperm(len(Xd), generator=torch.Generator().manual_seed(0)); tr, te = perm[:1300], perm[1300:]

class Square(nn.Module):
    def forward(self, x): return x * x

def fit(act):
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(64, 32), act, nn.Linear(32, 10)); opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    for _ in range(300):
        loss = F.cross_entropy(net(Xd[tr]), yd[tr]); opt.zero_grad(); loss.backward(); opt.step()
    return (net(Xd[te]).argmax(1) == yd[te]).float().mean().item()

acc_relu, acc_sq = fit(nn.ReLU()), fit(Square())
print(f"test accuracy with ReLU {acc_relu:.3f} | with square activation {acc_sq:.3f}")
assert acc_sq > 0.9

Each multiplication of ciphertexts uses up part of a limited 'depth' budget, so encrypted models are kept shallow, and softmax or attention must be replaced with approximations. Rho et al. (2025) do this for transformers, using LoRA and Gaussian kernels.

### 4. The same MLP with TenSEAL and CKKS (NOT RUN here)

In [ ]:
# NOT RUN: needs `pip install tenseal`.
import tenseal as ts
ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=8192, coeff_mod_bit_sizes=[40, 21, 21, 21, 40])
ctx.global_scale = 2 ** 21; ctx.generate_galois_keys()

# Train the square-activation MLP above, then take its weights:
# W1, b1, W2, b2 = [p.detach().numpy() for p in net.parameters()]
def encrypted_forward(x):
    enc = ts.ckks_vector(ctx, x.tolist())             # the whole feature vector in one ciphertext
    h = enc.matmul(W1.T.tolist()) + b1.tolist()       # linear layer with plain weights
    h = h.square()                                    # one ciphertext-ciphertext multiplication
    out = h.matmul(W2.T.tolist()) + b2.tolist()
    return out.decrypt()
# Time 20 encrypted predictions against 20 plain ones, check the predictions agree, and report the slowdown.

### TODO
1. Rerun section 2 with 512-bit primes. How does the time per patient change?
2. In section 2 the server sees no data, but the patient learns the exact score. What could a patient learn about the model by sending many queries?
3. Run the TenSEAL cell and compare its slowdown with the toy scheme. Why is vector packing such a large gain?